# One-Hot Encoding for Text

**Goal:** convert text into tokens, build a vocabulary, and encode each word as a one-hot vector.

**Flow:** Data -> Tokens -> Flat word list -> 2D input -> Encoder -> Encoded sentences

## Step 1: Import tools

- `re` cleans/splits text into words.
- `OneHotEncoder` creates one-hot vectors.
- `numpy` helps reshape data into 2D form.

In [123]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import re

## Step 2: Define sample data

- `doc` is the input corpus.
- Small examples make the encoding steps easy to revise.

In [124]:
doc = [
    "SDE loves coding",
    "SDE loves AI",
    "SDE codes in Python",
    "SDE codes AI"
]

print(doc)

['SDE loves coding', 'SDE loves AI', 'SDE codes in Python', 'SDE codes AI']


## Step 3: Tokenize each sentence

- `re.findall(r'\\b\\w+\\b', ...)` extracts words only.
- Lowercasing avoids duplicate forms like `SDE` and `sde`.

**Alternative:** use `.split()` for very simple text, or `nltk.word_tokenize()` for richer tokenization.

In [125]:
tokens = []

for sentence in doc:
    sentence_tokens = re.findall(r'\b\w+\b', sentence.lower())
    tokens.append(sentence_tokens)

print("Tokenized sentences:")
print(tokens)

Tokenized sentences:
[['sde', 'loves', 'coding'], ['sde', 'loves', 'ai'], ['sde', 'codes', 'in', 'python'], ['sde', 'codes', 'ai']]


## Step 3.5: Convert tokens to 2D word input

- `all_words` flattens the token list.
- Each word becomes `[word]` because the encoder needs **2D input**.
- This is word-based encoding, not position-based encoding.

**Improvement:** this is clearer than reshaping the full `tokens` list directly.

In [126]:
all_words = [[word] for sentence in tokens for word in sentence]

print("Single-column 2D word list:")
print(all_words)

Single-column 2D word list:
[['sde'], ['loves'], ['coding'], ['sde'], ['loves'], ['ai'], ['sde'], ['codes'], ['in'], ['python'], ['sde'], ['codes'], ['ai']]


## Step 4: Fit the encoder

- `OneHotEncoder` expects 2D input.
- Each row in `all_words` contains one word.
- This lets the encoder learn the full vocabulary once.

In [127]:
encoder = OneHotEncoder(sparse_output=False)
encoder.fit(all_words)

print("Encoder fitted on all words.")

Encoder fitted on all words.


## Step 5: Inspect learned vocabulary

- `encoder.categories_[0]` stores the unique words.
- Encoding the vocabulary once helps verify the mapping clearly.

**Alternative:** build a manual word-to-index dictionary and create vectors yourself.

In [128]:
vocabulary = encoder.categories_[0]
vocab_2d = np.array(vocabulary).reshape(-1, 1) # Reshape to 2D for the encoder
vocab_vectors = encoder.transform(vocab_2d)

print("Vocabulary:", vocabulary)
print("\nWord -> One-hot vector")
for word, vector in zip(vocabulary, vocab_vectors):
    print(f"{word:>6} -> {vector}")

Vocabulary: ['ai' 'codes' 'coding' 'in' 'loves' 'python' 'sde']

Word -> One-hot vector
    ai -> [1. 0. 0. 0. 0. 0. 0.]
 codes -> [0. 1. 0. 0. 0. 0. 0.]
coding -> [0. 0. 1. 0. 0. 0. 0.]
    in -> [0. 0. 0. 1. 0. 0. 0.]
 loves -> [0. 0. 0. 0. 1. 0. 0.]
python -> [0. 0. 0. 0. 0. 1. 0.]
   sde -> [0. 0. 0. 0. 0. 0. 1.]


## Step 6: Encode one sentence first

- Start with one example to verify the shape.
- Each row in the output is the one-hot vector for one token in the sentence.

In [129]:
example_sentence = tokens[0]
example_encoded = encoder.transform([[word] for word in example_sentence])

print("Example sentence tokens:", example_sentence)
print(example_encoded)

Example sentence tokens: ['sde', 'loves', 'coding']
[[0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]]


## Step 7: Encode all sentences

- The same fitted encoder is reused for every sentence.
- This keeps the vocabulary consistent across the dataset.

In [130]:
for sentence in tokens:
    encoded_sentence = encoder.transform([[word] for word in sentence])
    print("\nSentence:", sentence)
    print(encoded_sentence)


Sentence: ['sde', 'loves', 'coding']
[[0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]]

Sentence: ['sde', 'loves', 'ai']
[[0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0.]]

Sentence: ['sde', 'codes', 'in', 'python']
[[0. 0. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0.]]

Sentence: ['sde', 'codes', 'ai']
[[0. 0. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0.]]


## Revision notes

- One-hot encoding gives **one unique binary vector per word**.
- `OneHotEncoder` needs **2D input**, so words are wrapped as `[word]`.
- `tokens` keeps sentence structure; `all_words` flattens everything for fitting.
- `fit()` learns the vocabulary; `transform()` converts words into vectors.
- Use `encoder.transform([[word] for word in sentence])` to encode one sentence word by word.
